# Cylindrical equilibrium $\nabla\cdot T + f = 0$ (vibe 000073)

Reproducing the classic hand derivation of the balance equations for a
continuous medium in cylindrical coordinates.  We take the stress balance
$\nabla\cdot T + f = 0$ for a **symmetric** rank-2 stress $T$, represent it in
the cylindrical physical frame $\mathbf e_r,\mathbf e_\theta,\mathbf e_z$, and
let tender compute the divergence — differentiating the components **and** the
moving basis vectors together (via the connection
$\partial_\theta\mathbf e_r=\mathbf e_\theta$,
$\partial_\theta\mathbf e_\theta=-\mathbf e_r$).  The three scalar balance
equations fall out and match the standard textbook form.

In [ ]:
import tender as t
import tender.basis as tb
import tender.derivation as td
from IPython.display import Math, display

ws = t.Workspace()


def disp(*exprs, lhs=None):
    for e in exprs:
        s = e.latex() if hasattr(e, "latex") else str(e)
        display(Math((lhs + " = " if lhs else "") + s))

## 0. The cylindrical chart
Mapping $x=r\cos\theta,\ y=r\sin\theta,\ z=z$ over the world frame; $r\ge 0$.

In [ ]:
WCS = ws.wcs()
r, th, z = ws.coords("r", r"\theta", "z", nonneg=("r",))
cyl = ws.chart(WCS, [r, th, z], [r * t.cos(th), r * t.sin(th), z])
names = ["r", r"\theta", "z"]

## 1. The physical orthonormal frame $\mathbf e_i = \mathbf g_i/h_i$

In [ ]:
frame = cyl.physical_frame()
e = [frame.direction(k) for k in range(3)]
disp(*e)
print("orthonormal:", frame.is_orthonormal)

## 2. How the frame turns — the connection $\partial_j\mathbf e_i$
Only the $\theta$-derivatives are nonzero:
$\partial_\theta\mathbf e_r=\mathbf e_\theta$,
$\partial_\theta\mathbf e_\theta=-\mathbf e_r$.  These are what make the
curvilinear divergence differ from the flat one; tender derives them.

In [ ]:
def field_latex(components):
    parts = []
    for c, nm in zip(components, names):
        s = td.simplify_scalars(c).latex()
        if s == "0":
            continue
        vec = rf"\mathbf{{e}}_{{{nm}}}"
        parts.append(vec if s == "1" else rf"\left({s}\right)\,{vec}")
    return " + ".join(parts).replace("+ -", "-") or "0"


for i, j, lbl in [(0, 1, r"\partial_\theta \mathbf e_r"),
                  (1, 1, r"\partial_\theta \mathbf e_\theta"),
                  (0, 0, r"\partial_r \mathbf e_r")]:
    display(Math(lbl + " = " + field_latex(cyl.connection_coefficients(i, j))))

## 3. The stress field in the frame  (eq 4)
$T$ is an abstract **symmetric** rank-2 field; `expand_in_basis` writes it as
$\sum_{ij} T_{ij}\,\mathbf e_i\otimes\mathbf e_j$, the components inheriting both
the field-nature (so $\nabla$ can differentiate them) and the symmetry (so
$T_{\theta r}$ folds to $T_{r\theta}$).

In [ ]:
T = ws.field("T", 2, symmetric=True)  # T_ij = T_ji
T_cyl = td.canonicalize(td.unroll_sums(
    tb.expand_in_basis(T, frame, tb.Variance.Covariant)))
disp(T_cyl, lhs="T")

## 4. The divergence $\nabla\cdot T$, per frame component  (eq 7) — Route A
Hand the **abstract** field $T$ to `cyl.div`: it expands $T$ in the frame under
the hood and differentiates components **and** moving basis vectors.
`cyl.components` surfaces the three scalar components — the $\theta$-component
carries the classic $2T_{r\theta}/r$ shear term.

In [ ]:
div_r, div_th, div_z = cyl.components(cyl.div(T))
for nm, c in zip(names, (div_r, div_th, div_z)):
    display(Math(rf"(\nabla\cdot T)_{{{nm}}} = " + c.latex()))

`cyl.components(T)` on the rank-2 field itself returns the physical component
**matrix** $T_{ij} = \mathbf e_i \cdot T \cdot \mathbf e_j$ (vibe 000074), with the
symmetry already folded — the $(\theta,r)$ entry *is* $T_{r\theta}$.  And since
`algebraic_eq` sees through fraction shapes, the textbook cross-check is direct:
no manual reshaping of $\frac{a}{r}+\frac{b}{r}$ vs $\frac{a+b}{r}$.

In [ ]:
Tc = cyl.components(T)  # 3x3 physical component matrix e_i . T . e_j
display(Math(r"T_{ij} = \begin{pmatrix}"
             + r" \\ ".join(" & ".join(Tc[i][j].latex() for j in range(3))
                            for i in range(3))
             + r"\end{pmatrix}"))

d = td.partial
assert td.algebraic_eq(
    div_th,
    d(Tc[0][1], r) + d(Tc[1][1], th) / r + d(Tc[1][2], z) + 2 * Tc[0][1] / r)
print("θ-equation matches the textbook form ✓")

## 5. The balance law  $\nabla\cdot T + f = 0$  (eq 8)
The body force is a vector field; `cyl.components` surfaces $f_r,f_\theta,f_z$.
The balance equations are $(\nabla\cdot T)_i + f_i = 0$.

In [ ]:
f = ws.field("f", 1)
f_r, f_th, f_z = cyl.components(f)
for nm, dv, fi in zip(names, (div_r, div_th, div_z), (f_r, f_th, f_z)):
    display(Math(td.simplify_scalars(dv + fi).latex()
                 + r"\quad(\mathbf e_{" + nm + r"})"))

## 6. Axisymmetric, no shear, radial load only  (eq 9)
Assume $T$ and $f$ depend on $r$ only, with no shear and $f_\theta=f_z=0$.  The
$\theta$- and $z$-equations vanish; only the radial one survives:
$\partial_r T_{rr} + (T_{rr}-T_{\theta\theta})/r + f_r = 0$.

In [ ]:
T_r = ws.field("T", 2, deps=[r], symmetric=True)
f_rr = cyl.components(ws.field("f", 1, deps=[r]))[0]
radial = td.simplify_scalars(cyl.components(cyl.div(T_r))[0] + f_rr)
disp(radial, lhs=r"\partial_r T_{rr} + \tfrac{T_{rr}-T_{\theta\theta}}{r} + f_r")

## 7. The boiler formula  (eq 10)
A pipe of inner radius $R$, thickness $d\ll R$, internal pressure $p$, no body
force.  $T_{rr}$ runs from $-p$ at $r=R$ to $0$ at $r=R+d$, so
$\partial_r T_{rr}\approx p/d$; with $r/d\gg1$ that term dominates
$T_{rr}\in[-p,0]$, giving the hoop stress $T_{\theta\theta}\approx Rp/d$.